#### Q1. Embedding a query

In [22]:
import numpy as np
from tqdm import tqdm
from embedder import Embedder
embedder = Embedder()

In [23]:
query = "How does approximate nearest neighbor search work?"
v_query = embedder.encode(query)

print("=" * 60)
print("Q1. Embedding a query")
print("=" * 60)
print(f"Query: {query}")
print(f"Vector shape: {v_query.shape}")
print(f"First value (v[0]): {v_query[0]:.4f}")
print()

Q1. Embedding a query
Query: How does approximate nearest neighbor search work?
Vector shape: (384,)
First value (v[0]): -0.0206



In [24]:

from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Loaded {len(documents)} documents")
print()

Loaded 72 documents



#### Q2. Cosine similarity

In [25]:
target_file = "02-vector-search/lessons/07-sqlitesearch-vector.md"
target_doc = None
for doc in documents:
    if doc["filename"] == target_file:
        target_doc = doc
        break

v_doc = embedder.encode(target_doc["content"])
cosine_sim = v_query.dot(v_doc)

print("=" * 60)
print("Q2. Cosine similarity")
print("=" * 60)
print(f"File: {target_file}")
print(f"Cosine similarity: {cosine_sim:.4f}")
print()

Q2. Cosine similarity
File: 02-vector-search/lessons/07-sqlitesearch-vector.md
Cosine similarity: 0.3611



#### Q3. Chunking and search by hand

In [26]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")

# Embed all chunks
chunk_texts = [chunk["content"] for chunk in chunks]
chunk_vectors = embedder.encode_batch(chunk_texts)
X = np.array(chunk_vectors)

# Score query against all chunks
scores = X.dot(v_query)

# Find highest scoring chunk
best_idx = np.argmax(scores)
best_chunk = chunks[best_idx]

print("=" * 60)
print("Q3. Chunking and search by hand")
print("=" * 60)
print(f"Best score: {scores[best_idx]:.4f}")
print(f"Best chunk filename: {best_chunk['filename']}")
print(f"Best chunk start: {best_chunk['start']}")
print()


Total chunks: 295
Q3. Chunking and search by hand
Best score: 0.6489
Best chunk filename: 02-vector-search/lessons/07-sqlitesearch-vector.md
Best chunk start: 1000



#### Q4. Vector search with minsearch

In [27]:
import minsearch

# Index chunks with vector search
index = minsearch.VectorSearch(
    text_fields=["content"],
    vector_fields=["content_vector"],
    keyword_fields=["filename", "start"]
)

# Prepare documents with pre-computed vectors
for i, chunk in enumerate(chunks):
    chunk["content_vector"] = X[i].tolist()

index.fit(chunks)

query_q4 = "What metric do we use to evaluate a search engine?"
v_q4 = embedder.encode(query_q4)

results_vector = index.search(v_q4, num_results=5)

print("=" * 60)
print("Q4. Vector search with minsearch")
print("=" * 60)
print(f"Query: {query_q4}")
print(f"First result filename: {results_vector[0]['filename']}")
print()

TypeError: VectorSearch.__init__() got an unexpected keyword argument 'text_fields'

####  Q5. Text search vs vector search

In [ ]:
# Text search index
text_index = minsearch.Index(
    text_fields=["content"],
    keyword_fields=["filename", "start"]
)
text_index.fit(chunks)

query_q5 = "How do I store vectors in PostgreSQL?"
v_q5 = embedder.encode(query_q5)

vector_results_q5 = index.search(v_q5, num_results=5)
text_results_q5 = text_index.search(query_q5, num_results=5)

vector_files_q5 = {r["filename"] for r in vector_results_q5}
text_files_q5 = {r["filename"] for r in text_results_q5}

only_vector = vector_files_q5 - text_files_q5

print("=" * 60)
print("Q5. Text search vs vector search")
print("=" * 60)
print(f"Query: {query_q5}")
print(f"Vector results: {vector_files_q5}")
print(f"Text results: {text_files_q5}")
print(f"In vector but NOT in text: {only_vector}")
print()

#### Q6. Hybrid search with RRF

In [ ]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", 0))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

query_q6 = "How do I give the model access to tools?"
v_q6 = embedder.encode(query_q6)

vector_results_q6 = index.search(v_q6, num_results=10)
text_results_q6 = text_index.search(query_q6, num_results=10)

hybrid_results = rrf([vector_results_q6, text_results_q6])

print("=" * 60)
print("Q6. Hybrid search with RRF")
print("=" * 60)
print(f"Query: {query_q6}")
print(f"Top RRF result filename: {hybrid_results[0]['filename']}")
print()

In [ ]:
print("=" * 60)
print("ANSWERS SUMMARY")
print("=" * 60)
print(f"Q1. First value v[0]: {v_query[0]:.4f}")
print(f"Q2. Cosine similarity: {cosine_sim:.4f}")
print(f"Q3. Highest-scoring chunk file: {best_chunk['filename']}")
print(f"Q4. First vector search result: {results_vector[0]['filename']}")
print(f"Q5. In vector but not text: {only_vector}")
print(f"Q6. Top RRF result: {hybrid_results[0]['filename']}")
print("=" * 60)